In [6]:
import pandas as pd
from ipyfilechooser import FileChooser
from IPython.display import display
import os


In [10]:

dpath = "//10.69.168.1/crnldata/forgetting/Carla/"

print("🔍 Sélectionnez votre fichier Excel à analyser:")

# Sélecteur de fichier Excel directement
fc1 = FileChooser(dpath, filter_pattern='*.csv', 
                  title="<b>Sélectionnez votre fichier Excel à analyser</b>")
display(fc1)

def update_file(chooser):
    global fichier_choisi, dpath
    fichier_choisi = chooser.selected
    # Mémoriser le dossier pour la prochaine fois
    dpath = os.path.dirname(fichier_choisi)
    %store dpath
    print(f"✅ Fichier sélectionné: {os.path.basename(fichier_choisi)}")
    return

fc1.register_callback(update_file)

🔍 Sélectionnez votre fichier Excel à analyser:


FileChooser(path='\\10.69.168.1\crnldata\forgetting\Carla', filename='', title='<b>Sélectionnez votre fichier …

In [ ]:

# Lire le fichier et analyser les épisodes
if 'fichier_choisi' not in globals():
    print("❌ Veuillez d'abord exécuter la cellule 1 et sélectionner un fichier")
else:
    print(f"📊 Lecture et analyse du fichier: {os.path.basename(fichier_choisi)}")
    print("-" * 60)
    
    # Lire le fichier (CSV ou Excel)
    if fichier_choisi.endswith('.csv'):
        df = pd.read_csv(fichier_choisi)
    else:
        df = pd.read_excel(fichier_choisi)
    
    print(f"📋 Données lues: {len(df)} lignes")
    print(f"🔍 Colonnes trouvées: {list(df.columns)}")
    print()
    
    # Vérifier les colonnes nécessaires
    if 'label' not in df.columns or 'duration' not in df.columns:
        print("❌ Le fichier doit contenir les colonnes 'label' et 'duration'")
    else:
        # ANALYSE DES ÉPISODES (vectorisé, sans boucle)
        print("⚡ Regroupement des épisodes consécutifs...")
        
        # 1. Identifier les changements de label (vectorisé)
        changements = df['label'] != df['label'].shift(1)
        
        # 2. Créer des groupes d'épisodes
        df['groupe_episode'] = changements.cumsum()
        
        # 3. Regrouper par épisode et calculer les totaux
        episodes = df.groupby('groupe_episode').agg({
            'label': 'first',
            'duration': 'sum'
        }).reset_index()
        
        # 4. Renommer et réorganiser
        episodes = episodes.rename(columns={
            'groupe_episode': 'episode',
            'duration': 'duree_totale'
        })
        
        # 5. Calculer début et fin
        episodes['fin'] = episodes['duree_totale'].cumsum()
        episodes['debut'] = episodes['fin'] - episodes['duree_totale']
        
        # 6. Réorganiser les colonnes
        episodes = episodes[['episode', 'label', 'duree_totale', 'debut', 'fin']]
        
        # AFFICHAGE DES RÉSULTATS
        print("✅ ÉPISODES DE SOMMEIL:")
        print(episodes.to_string(index=False))
        print()
        
        # STATISTIQUES RAPIDES
        print("📊 RÉSUMÉ:")
        print(f"   • Nombre total d'épisodes: {len(episodes)}")
        print(f"   • Durée totale: {episodes['duree_totale'].sum()}")
        print(f"   • Phases détectées: {', '.join(episodes['label'].unique())}")
        
        # Compter les épisodes par phase
        count_by_phase = episodes['label'].value_counts()
        print("   • Nombre d'épisodes par phase:")
        for phase, count in count_by_phase.items():
            total_duree = episodes[episodes['label'] == phase]['duree_totale'].sum()
            print(f"     - {phase}: {count} épisodes (durée totale: {total_duree})")
        print()
        
        # SAUVEGARDE AUTOMATIQUE
        # Créer le nom du fichier de sortie
        dossier = os.path.dirname(fichier_choisi)
        nom_original = os.path.basename(fichier_choisi)
        nom_sans_ext = os.path.splitext(nom_original)[0]
        fichier_sortie = os.path.join(dossier, f"{nom_sans_ext}_episodes.xlsx")
        
        # Sauvegarder
        episodes.to_excel(fichier_sortie, index=False, sheet_name='Episodes')
        print(f"💾 Résultats sauvegardés dans: {os.path.basename(fichier_sortie)}")
        
        # Stocker les résultats pour utilisation ultérieure
        episodes_resultats = episodes

📊 Lecture et analyse du fichier: EphyViewer_AutoScor5.csv
------------------------------------------------------------
📋 Données lues: 34636 lignes
🔍 Colonnes trouvées: ['label', 'duration', 'time']

⚡ Regroupement des épisodes consécutifs...
✅ ÉPISODES DE SOMMEIL:
 episode  label  duree_totale  debut    fin
       1 Wake_1          1620      0   1620
       2 NREM_2             5   1620   1625
       3 Wake_1             5   1625   1630
       4 NREM_2             5   1630   1635
       5 Wake_1          1650   1635   3285
       6 NREM_2           455   3285   3740
       7  REM_3            80   3740   3820
       8 Wake_1            30   3820   3850
       9 NREM_2           400   3850   4250
      10 Wake_1            30   4250   4280
      11 NREM_2           185   4280   4465
      12 Wake_1            15   4465   4480
      13 NREM_2           230   4480   4710
      14  REM_3            80   4710   4790
      15 NREM_2           605   4790   5395
      16 Wake_1            20 

In [18]:
# =============================================================================
# CELLULE 3 : STATISTIQUES DÉTAILLÉES ET SAUVEGARDE (corrigée)
# =============================================================================

if 'episodes' not in globals():
    print("❌ Veuillez d'abord exécuter les cellules 1 et 2")
else:
    print("📊 CALCUL DES STATISTIQUES DÉTAILLÉES")
    print("=" * 60)
    
    # Calculer les statistiques par label
    stats_par_label = episodes.groupby('label').agg({
        'duree_totale': ['sum', 'mean', 'count']
    }).round(2)
    
    # Aplatir les colonnes
    stats_par_label.columns = ['duree_totale', 'duree_moyenne', 'nb_episodes']
    stats_par_label = stats_par_label.reset_index()
    
    # Durée totale de l'enregistrement (en secondes)
    duree_totale_enreg = episodes['duree_totale'].sum()
    if duree_totale_enreg == 0:
        raise ValueError("Durée totale d'enregistrement égale à 0 — impossible de normaliser.")
    
    # Calculs des pourcentages et conversions
    print("🔢 RÉSUMÉ GLOBAL:")
    print(f"   • Durée totale d'enregistrement: {duree_totale_enreg} secondes")
    print(f"   • Soit {duree_totale_enreg/3600:.2f} heures")
    print(f"   • Nombre total d'épisodes: {len(episodes)}")
    print(f"   • Phases détectées: {', '.join(episodes['label'].unique())}")
    print()
    
    # Affichage des stats détaillées
    print("📈 STATISTIQUES PAR PHASE:")
    print("-" * 95)
    print(f"{'Phase':<10} {'Episodes':<10} {'Durée Totale (s)':<17} {'Durée Moy (s)':<14} {'% sur 48h':<10} {'% du total':<11} {'Min/h':<10}")
    print("-" * 95)
    
    for _, row in stats_par_label.iterrows():
        phase = row['label']
        nb_eps = int(row['nb_episodes'])
        duree_tot = row['duree_totale']     # en secondes
        duree_moy = row['duree_moyenne']    # en secondes
        
        # Pourcentage par rapport à 48h
        pct_48h = (duree_tot / 172800) * 100  # 48h = 172800 sec
        
        # Pourcentage par rapport au total de l’enregistrement
        pct_du_total = (duree_tot / duree_totale_enreg) * 100
        
        # ✅ Minutes par heure d’enregistrement
        min_par_heure = (duree_tot / duree_totale_enreg) * 60
        
        print(f"{phase:<10} {nb_eps:<10} {duree_tot:<17.0f} {duree_moy:<14.2f} {pct_48h:<10.2f} {pct_du_total:<11.2f} {min_par_heure:<10.2f}")
    
    print("-" * 95)
    print()
    
    # Tableau récapitulatif
    print("📋 TABLEAU RÉCAPITULATIF:")
    stats_recap = stats_par_label.copy()
    stats_recap['pct_48h'] = (stats_recap['duree_totale'] / 172800 * 100).round(2)
    stats_recap['pct_du_total'] = (stats_recap['duree_totale'] / duree_totale_enreg * 100).round(2)
    stats_recap['minutes_par_heure'] = (stats_recap['duree_totale'] / duree_totale_enreg * 60).round(2)
    
    # Renommer pour affichage
    stats_recap_display = stats_recap.rename(columns={
        'label': 'Phase',
        'nb_episodes': 'Nb_Episodes',
        'duree_totale': 'Duree_Tot_sec',
        'duree_moyenne': 'Duree_Moy_sec',
        'pct_48h': '%_sur_48h',
        'pct_du_total': '%_du_total',
        'minutes_par_heure': 'Min_par_heure'
    })
    
    print(stats_recap_display.to_string(index=False))
    print()
    
    # SAUVEGARDE AUTOMATIQUE avec statistiques
    dossier = os.path.dirname(fichier_choisi)
    nom_original = os.path.basename(fichier_choisi)
    nom_sans_ext = os.path.splitext(nom_original)[0]
    fichier_sortie = os.path.join(dossier, f"{nom_sans_ext}_episodes.xlsx")
    
    with pd.ExcelWriter(fichier_sortie, engine='openpyxl') as writer:
        # Feuille 1: Episodes
        episodes.to_excel(writer, sheet_name='Episodes', index=False)
        
        # Feuille 2: Statistiques détaillées
        stats_recap_display.to_excel(writer, sheet_name='Statistiques', index=False)
        
        # Feuille 3: Résumé simple
        resume_simple = pd.DataFrame({
            'Métrique': [
                'Durée totale (sec)', 
                'Durée totale (h)', 
                'Nombre d\'épisodes', 
                'Nombre de phases'
            ],
            'Valeur': [
                duree_totale_enreg, 
                round(duree_totale_enreg/3600, 2), 
                len(episodes), 
                len(stats_recap)
            ]
        })
        resume_simple.to_excel(writer, sheet_name='Resume', index=False)
    
    print(f"💾 Résultats sauvegardés dans: {os.path.basename(fichier_sortie)}")
    print("   📊 3 feuilles créées: Episodes, Statistiques, Resume")
    
    # Stocker les résultats pour réutilisation
    episodes_resultats = episodes
    stats_resultats = stats_recap_display


📊 CALCUL DES STATISTIQUES DÉTAILLÉES
🔢 RÉSUMÉ GLOBAL:
   • Durée totale d'enregistrement: 173180 secondes
   • Soit 48.11 heures
   • Nombre total d'épisodes: 510
   • Phases détectées: Wake_1, NREM_2, REM_3

📈 STATISTIQUES PAR PHASE:
-----------------------------------------------------------------------------------------------
Phase      Episodes   Durée Totale (s)  Durée Moy (s)  % sur 48h  % du total  Min/h     
-----------------------------------------------------------------------------------------------
NREM_2     229        58780             256.68         34.02      33.94       20.36     
REM_3      130        8685              66.81          5.03       5.02        3.01      
Wake_1     151        105715            700.10         61.18      61.04       36.63     
-----------------------------------------------------------------------------------------------

📋 TABLEAU RÉCAPITULATIF:
 Phase  Duree_Tot_sec  Duree_Moy_sec  Nb_Episodes  %_sur_48h  %_du_total  Min_par_heure
NREM_2 